# 预测伪随机数：用 Z3 把 `rand()` 的种子反推出来

## 这一章要做什么

伪随机数看着挺乱，可生成的办法只要你知道，后面的值就能算出来。这一章拿 Visual C++ 的 `rand()` 开刀。

- `rand()` 底下是线性同余生成器（LCG）
- 输出只要拿到几个，内部状态（种子）就能用 Z3 反推出来
- 种子一到手，下一个值也就跟着出来了

两个主角：`BitVec`（定长的比特串）和 `URem`（无符号取余）。

## 线性同余生成器（LCG）

**线性同余生成器**（Linear Congruential Generator, LCG）是造伪随机数的老办法之一。下一个状态由下面这条递推式给出。

$$
X_{n+1} = \left(A \times X_n + B\right) \bmod M
$$

$A$, $B$, $M$ 都是常数，一般要满足几条条件。

- $M > A$
- $M > B$
- $A > 0$
- $B \geq 0$

$M = 2^{32}$ 的时候，这一式子就是 32 位整数的乘加，再把溢出来的那截扔掉。

## Visual C++ 的 `rand()`

Visual C++ 的 `rand()` 走的就是线性同余这条路。源码长这样。

```c
static long holdrand = 1L;

int __cdecl rand(void)
{
    return (((holdrand = holdrand * 214013L + 2531011L) >> 16) & 0x7FFF);
}

void __cdecl srand(unsigned int seed)
{
    holdrand = (long)seed;
}
```

它干了两件事。

1. 把内部状态 `holdrand` 往前推一步

$$
\text{holdrand} = (\text{holdrand} \times 214013 + 2531011) \bmod 2^{32}
$$

2. 取高 16 位，当成 0～32767（`0x7FFF`）的整数返回

输出拿到的只是内部状态的**一部分**。正因为如此，从输出倒推状态就成了一个问题：信息够不够。

接下来我们自己写一个行为一模一样的 Python 类。

In [1]:
class PseudoRandom:
    """和 Visual C++ 的 rand() 行为相同的伪随机数"""

    def __init__(self, seed=1):
        self._holdrand = seed

    def srand(self, seed: int):
        self._holdrand = seed

    def rand(self) -> int:
        self._holdrand = (self._holdrand * 214013 + 2531011) & 0xFFFFFFFF
        return (self._holdrand >> 16) & 0x7FFF

## 先生成 8 个随机数

种子取 `1792`，连出 8 个 `rand() % 100`（0～99）。第 9 个先攥着，最后拿来对答案。

In [2]:
rand = PseudoRandom(1792)
numbers = [rand.rand() % 100 for _ in range(8)]
print(numbers)
print(rand.rand() % 100)

[90, 79, 97, 7, 20, 77, 17, 89]
16


## 用 Z3 反推种子

`numbers` 里只有 8 个数，第 9 个猜得出来吗？

内部状态是 32 位，那就用一个 `BitVec('seed', 32)` 变量来当它。然后把 `rand()` 那几行计算原封不动写成约束。

1. `states[0]` 是“种子走了一步之后的状态”，`states[1]` 再往后一步……一个挨一个排下去
2. 相邻两个状态必须满足 `state_{i+1} = state_i * a + b`（都在 32 位里绕，乘出来溢出的部分自动就没了）
3. 每个状态取出来的 `(state >> 16) & 0x7FFF`，拿 `mod` 除一下取余，得跟观测到的 `numbers` 对上

只看得到除以 `mod` 的余数，状态本身未必能定成一个。“能生出同一串数的种子”却总归找得到。

In [3]:
from z3 import *


def find_seed_and_next(numbers, mod, a=214013, b=2531011):
    """从观测到的输出序列，求出种子和下一个输出"""
    seed = BitVec('seed', 32)
    states = [BitVec(f's_{i}', 32) for i in range(len(numbers) + 1)]
    outputs = [URem((s >> 16) & 0x7FFF, mod) for s in states]

    solver = Solver()

    # 1. 状态的转移（LCG）
    for prev, cur in zip([seed] + states[:-1], states):
        solver.add(prev * a + b == cur)

    # 2. 和观测到的输出一致
    for out, observed in zip(outputs, numbers):
        solver.add(out == observed)

    if solver.check() != sat:
        return None

    model = solver.model()
    return model[seed].as_long(), model.eval(outputs[-1]).as_long()


seed_val, next_val = find_seed_and_next(numbers, mod=100)
print(f'seed = {seed_val}')
print(f'next = {next_val}')

seed = 1792
next = 16


原来的种子 `1792` 被揪出来了，下一个值算出来是 `16`。真正的第 9 个数也是 `16`（翻上去看看那个格子的输出）。

拿这个种子，把数列重新生成一遍。

In [4]:
rand = PseudoRandom(seed_val)
[rand.rand() % 100 for _ in range(9)]

[90, 79, 97, 7, 20, 77, 17, 89, 16]

9 个数一个一个排出来，和原来一模一样。猜中了。

不过话说回来，`mod 100` 这一步把信息截掉了一截，**满足条件的种子未必只有一个**。这道题里 `1792` 和 `2147485440` 都过关，两个种子给出的数列也完全一样。输出拿到的信息越少，候选就越多。

## 换个玩法：让骰子连出 10 个 6

骰子是用 `rand() % 6 + 1` 掷的。能不能找一个种子，让它连着十次都是 6？

6 对应的是 `rand() % 6 == 5`，所以观测值直接传 `[5] * 10` 就行。

In [5]:
seed_val, next_val = find_seed_and_next([5] * 10, mod=6)
print(seed_val)

rand = PseudoRandom(seed_val)
[rand.rand() % 6 for _ in range(11)]

3689596556


[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0]

连着十次都是 `5`（也就是骰子掷出 6）的种子，找到了。

只是别忘了，放进条件里的**只有前 10 次**。第 11 次没管，所以它不一定还是 `5`（这一跑出来是 `0`）。真要“11 次连出 6”，把那十一项 `[5] * 11` 都塞进条件就行。

还有一点：满足条件的种子不止一个，每跑一次都可能撞见不同的种子。

## `BitVec` 和 `URem`

`BitVec` 是 Z3 里**定长的比特串**类型。

- `BitVec('x', 32)` 就当 32 位整数使
- 乘啊加啊溢出多少，扔多少（跟 C 的无符号整数一个脾气）
- `&`, `|`, `^`, `>>`, `<<` 这些位运算都能用

`URem(a, b)` 是**无符号取余**，也就是 `a % b`。`BitVec` 身上没有符号，取余就得喊 `URem()`，不能直接用 `%`。

这两样在手，`rand()` 的代码照着抄就行。

| C 的代码 | Z3 的表达式 |
|---|---|
| `holdrand * 214013 + 2531011` | `prev * 214013 + 2531011` |
| 32 位溢出的部分丢掉 | `BitVec` 的计算（自动） |
| `holdrand >> 16` | `s >> 16` |
| `& 0x7FFF` | `& 0x7FFF` |
| `% mod` | `URem(..., mod)` |

## 小结

- 生成式知道，就能从输出倒推内部状态，办法是把生成式原样写进 `BitVec`
- 输出只露出状态的一部分时，种子候选可能不止一个
- 有了反推出来的种子，下一个值就能算
- 像“连续 10 次都是 6”这种要求，也无非是当观测值传进去，让 Z3 去找种子